In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import mplcursors

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

In [ ]:
DATA_DIR = Path("/home/rauls/Desktop/VirgoBRET/cut_parquet_data")
PARQUET_FILES = sorted(DATA_DIR.glob("**/*/qtransform_features.parquet"))

RANDOM_STATE = 42

PERPLEXITY = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
EARLY_EXAGGERATION = [10, 12, 14, 20, 100, 300, 1000]
PCA_COMPONENTS = [2, 10, 20, 40]
METRIC = ['euclidean', 'manhattan', 'cosine']

'''
https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.manhattan_distances.html 
pairwise.manhattan_distances -> Compute the L1 distances between the vectors in X and Y.
pairwise.paired_manhattan_distances -> Compute the paired L1 distances between X and Y.
'''

In [ ]:
if len(PARQUET_FILES) == 0:
    raise FileNotFoundError(
        f"No parquet files found inside: {DATA_DIR}"
    )

print(f"Found {len(PARQUET_FILES)} parquet files.")

dataframes = []

for file in PARQUET_FILES:
    print(f"Loading: {file}")
    df = pd.read_parquet(file)
    df["source_file"] = file.name
    df["source_path"] = str(file)

    dataframes.append(df)

df_all = pd.concat(dataframes,ignore_index=True)

feature_columns = [col for col in df_all.columns if col.startswith("feature_")]
if len(feature_columns) == 0:
    raise ValueError("No feature_* columns found.")

print(f"Number of features: {len(feature_columns)}")

X = df_all[feature_columns].to_numpy(dtype=np.float64)

print("Feature matrix:")
print(f"X.shape = {X.shape}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Scaled matrix: {X_scaled.shape}") 

In [ ]:
for p in PERPLEXITY:
    tsne = TSNE(n_components=2,perplexity=p,init="pca",learning_rate="auto",random_state=RANDOM_STATE)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(12, 10))
    plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)
    plt.title(f"Perplexity={p}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
for m in METRIC:
    tsne = TSNE(n_components=2,perplexity=40,init="pca",learning_rate="auto",random_state=RANDOM_STATE, metric=m)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(12, 10))
    plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)
    plt.title(f"Perplexity = 40 | Metric = {m}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
for e in EARLY_EXAGGERATION:
    tsne = TSNE(n_components=2,perplexity=40,early_exaggeration=e,init="pca",learning_rate="auto",random_state=RANDOM_STATE)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(12, 10))
    plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)
    plt.title(f"Perplexity = 40 | Early Exaggeration = {e}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
PCA_COMPONENTS = 20

n_components = min(PCA_COMPONENTS,X_scaled.shape[0] - 1,X_scaled.shape[1])
pca = PCA(n_components=n_components,random_state=RANDOM_STATE)

X_pca = pca.fit_transform(X_scaled)
explained_variance = np.sum(pca.explained_variance_ratio_)

print(f"PCA components: {n_components}")
print(f"Explained variance: {explained_variance:.4f}")

In [ ]:
for p in PERPLEXITY:
    tsne = TSNE(n_components=2,perplexity=p,init="pca",learning_rate="auto",random_state=RANDOM_STATE)
    X_tsne = tsne.fit_transform(X_pca)

    plt.figure(figsize=(12, 10))
    plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)
    plt.title(f"Perplexity={p}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
for m in METRIC:
    tsne = TSNE(n_components=2,perplexity=40,init="pca",learning_rate="auto",random_state=RANDOM_STATE, metric=m)
    X_tsne = tsne.fit_transform(X_pca)

    plt.figure(figsize=(12, 10))
    plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)
    plt.title(f"Perplexity = 40 | Metric = {m}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
for e in EARLY_EXAGGERATION:
    tsne = TSNE(n_components=2,perplexity=40,early_exaggeration=e,init="pca",learning_rate="auto",random_state=RANDOM_STATE)
    X_tsne = tsne.fit_transform(X_pca)

    plt.figure(figsize=(12, 10))
    plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)
    plt.title(f"Perplexity = 40 | Early Exaggeration = {e}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    
    plt.tight_layout()
    plt.show()

In [ ]:
%matplotlib widget

tsne = TSNE(n_components=2,perplexity=40,early_exaggeration=500,init="pca",learning_rate="auto",random_state=RANDOM_STATE, metric='manhattan')
X_tsne = tsne.fit_transform(X_pca)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(X_tsne[:, 0],X_tsne[:, 1],s=5,alpha=0.5)

cursor = mplcursors.cursor(scatter, hover=True)

@cursor.connect("add")
def on_hover(sel):
    sel.annotation.set_text(f"Index: {sel.index}")

plt.title(f"Perplexity = 40 | Early Exaggeration = 500 | PCA Components = {PCA_COMPONENTS} | Metric = manhattan")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
    
plt.tight_layout()
plt.show()